# Burn cost demo: try a model and save its recipe

Read the rows saved in 01. Define features, groups and special levels, then fit a Tweedie model.
This notebook writes a TOML configuration only. It never publishes the fitted model to a database.


In [ ]:
from pathlib import Path

PROJECT_ROOT = next(
    candidate
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (candidate / "pyproject.toml").is_file() and (candidate / "pricing_models").is_dir()
)

import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from superglm import (
    Categorical,
    Numeric,
    OrderedCategorical,
    Spline,
    SuperGLM,
    Tweedie,
    collapse_levels,
)

from pricing_pipeline.models.config import ValidationSplitConfig
from pricing_pipeline.notebook import (
    ModelRecipe,
    PricingDataset,
    PricingModelSpec,
)

MODEL_DIR = PROJECT_ROOT / "pricing_models/burn_cost_demo"
DATASET_PATH = MODEL_DIR / ".local" / "dataset.joblib"

## Load the dataset from 01

Use the saved rows in their existing order. If you accept an enrichment or
source-data change here, put it in 01 and save the updated dataset before using 03.


In [ ]:
dataset = PricingDataset.load(DATASET_PATH)
df = dataset.df
display({"Rows": len(df), "Columns": len(df.columns)})
display(df.head())

## Define the features

`region` groups East and West together. `bonus_malus` smooths the ordered levels and keeps Unknown as a special level.
`driver_age` uses a cubic regression spline with quantile knots. Edit this dictionary to try other choices.


In [ ]:
features = {
    "region": Categorical(
        base="North",
        grouping=collapse_levels(df.region, groups={"EastWest": ["East", "West"]}),
    ),
    "bonus_malus": OrderedCategorical(
        order=["0", "1", "2", "3", "4"],
        specials=["Unknown"],
        base="0",
        basis=Spline("cr", k=3, knot_strategy="quantile"),
    ),
    "driver_age": Spline("cr", k=3, knot_strategy="quantile"),
}

## Choose the target and fitting settings

Burn cost is loss per exposure, so exposure is the weight. There is no offset in this example.
Notebook 03 will run the validation selected here.


In [ ]:
MODEL = PricingModelSpec(
    # Model identity.
    name="DEMO_BURN_COST",
    label="Burn cost workflow demo",
    model_type="burn_cost",
    deployment_slot="DEMO_BURN_COST_ONLY",
    # Data and feature columns.
    dataset=dataset,
    target="burn_cost",
    features=list(features),
    # Fit and validate.
    sample_weight_column="exposure",
    export_weight_column="exposure",
    fit_mode="fit_reml",
    validation=ValidationSplitConfig.kfold(n_splits=2, random_state=19),
)

model = SuperGLM(
    family=Tweedie(p=1.5),
    features=features,
    selection_penalty=0.0,
    retain_fit_state=False,
)

## Fit locally

Edit the setup above and rerun from there to try another configuration.


In [ ]:
X = df.loc[:, list(MODEL.features)]
y = df[MODEL.target]
sample_weight = None if MODEL.sample_weight_column is None else df[MODEL.sample_weight_column]
offset = None if MODEL.offset_column is None else df[MODEL.offset_column]

model.fit_reml(X, y, sample_weight=sample_weight, offset=offset)

## Inspect the fit

These predictions use the fitting data. Use held-out data to assess performance;
03 runs the validation specified above. Add plots and comparisons as needed.


In [ ]:
predictions = model.predict(X, offset=offset)
results = pd.DataFrame({"actual": y, "prediction": predictions})
display(model.summary())
display(model.relativities(with_se=False, centering="native"))
display(results.head())

In [ ]:
# Add your plots, comparisons and alternative fits here.


## Save the recipe for 03

This writes the feature definitions, groups, specials and fitting settings to `prototype.toml`.
It does not save the fitted coefficients there. Replacement is enabled for this demo, so rerunning updates the recipe with your current choices.
Notebook 03 loads these choices, fits and validates, then saves a model version to SQL.


In [ ]:
REPLACE_RECIPE = True  # Replace this demo's recipe with the current model settings.
recipe = ModelRecipe.from_model(model, spec=MODEL)
recipe.save(MODEL_DIR / "prototype.toml", replace=REPLACE_RECIPE)
print((MODEL_DIR / "prototype.toml").read_text())

Next: **03_model_training.ipynb** fits, validates and saves a version to SQL Server.